# Zeitreihe: Wie viele Fahrten kommen morgen?

**CRISP-DM-Block, Modeling-Phase (Block 05, Folie „Zeitreihenprognose schätzt einen
zukünftigen Wert")**

## Worum es geht

Das letzte der vier Grundverfahren-Notebooks. Wir sagen wieder eine Zahl vorher (wie
in Notebook 1), aber diesmal mit einem entscheidenden Unterschied: **die Zeit selbst ist
das zentrale Merkmal.** Wir wollen wissen, wie viele VeloCity-Fahrten an einem
bestimmten zukünftigen Tag stattfinden werden.

## Warum Zeitreihenprognose eine eigene Disziplin ist

Bei einer normalen Regression (Notebook 1) war es unproblematisch, die Daten zufällig in
Trainings- und Testmenge aufzuteilen. Bei einer Zeitreihe wäre das ein schwerer Fehler:
Wenn wir einen zufälligen Tag im März zum Training und einen zufälligen Tag im April zum
Testen verwenden, könnte das Modell versehentlich aus der *Zukunft* lernen, um die
*Vergangenheit* vorherzusagen — in der Praxis unmöglich. Deshalb gilt hier eine harte
Regel, dieselbe, die auch in eurem BINT-Notebook zur Forecasting-Übung gilt: **Die
Testdaten müssen zeitlich nach den Trainingsdaten liegen, ohne Ausnahme.**

## Der Plan dieses Notebooks

Wir bauen und vergleichen zwei Prognoseansätze für dieselbe Frage:
1. **Klassisch:** eine saisonale Basislinie — der Mittelwert der letzten vergleichbaren
   Wochentage, ganz ohne maschinelles Lernen
2. **Modern:** Gradient Boosting, ein Verfahren, das Wetter, Wochentag, Monat und
   Feiertage gleichzeitig einbezieht

Genau dieser Vergleich steht auch auf der Modeling-Folie in Block 05: *"Innerhalb jeder
Kategorie gibt es viele mögliche Algorithmen"* — hier wird das an einer Stelle konkret.

## Woher die Daten kommen

`ausleihe.csv` ist erfunden, `wetter.csv` und `feiertage.csv` sind echte historische
Daten für Würzburg (siehe `analytics/README.md`). Die eingebaute Korrelation zwischen
Tagesfahrten und Temperatur liegt bei r ≈ 0,79 — ein außergewöhnlich starkes Signal, das
wir hier gezielt nutzen werden.

## Lernziele

1. Tagesdaten aus Einzelfahrten aggregieren
2. Einen zeitlich korrekten Trainings-/Testsplit anlegen (kein Zufalls-Split!)
3. Eine saisonale Basislinie ohne maschinelles Lernen bauen
4. Ein modernes Verfahren (Gradient Boosting) mit mehreren Merkmalen trainieren
5. Beide mit dem MAPE (mittlerer absoluter prozentualer Fehler) fair vergleichen

## Schritt 1 — Bibliotheken importieren

Neu: `GradientBoostingRegressor` für den modernen Ansatz. Die MAPE-Funktion schreiben
wir selbst — genau wie im BINT-Notebook, denn `scikit-learn` hat zwar viele
Fehlermaße eingebaut, aber MAPE in Prozent ist am einfachsten direkt selbst
formuliert.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

pd.set_option("display.max_columns", 20)


def mape(y, yhat):
    """Mittlerer absoluter prozentualer Fehler - je kleiner, desto besser."""
    return float(np.mean(np.abs((y - yhat) / y)) * 100)


print("Bibliotheken geladen.")

## Schritt 2 — Daten laden und auf Tagesebene aggregieren

**TODO:**
1. Laden Sie `ausleihe.csv` (mit `startzeit` geparst), `wetter.csv` (mit `datum`
   geparst), `feiertage.csv` (mit `datum` geparst) und `veranstaltungen.csv` (mit
   `von` und `bis` geparst — Großveranstaltungen laufen über mehrere Tage, keinen
   einzelnen).
2. Legen Sie in `ausleihe` eine Spalte `datum` an (Datum ohne Uhrzeit).
3. Zählen Sie mit `.groupby("datum").size()` die Fahrten je Tag — das Ergebnis ist
   unsere Zielgröße, eine Zeile pro Tag.

In [ ]:
ausleihe = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/ausleihe.csv", parse_dates=["startzeit"])
wetter = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/wetter.csv", parse_dates=["datum"])
feiertage = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/feiertage.csv", parse_dates=["datum"])
veranstaltungen = pd.read_csv("https://raw.githubusercontent.com/swrobuts/velocity-fallstudie/main/analytics/veranstaltungen.csv", parse_dates=["von", "bis"])

ausleihe["datum"] = ...  # TODO

tage = ...  # TODO: ausleihe.groupby("datum").size(), dann .reset_index(name="fahrten")
tage = tage.sort_values("datum").reset_index(drop=True)

print(tage["datum"].min(), "bis", tage["datum"].max(), "-", len(tage), "Tage")
assert len(tage) > 1000, "Es sollten gut drei Jahre Tagesdaten sein"
tage.head()

## Schritt 3 — Merkmale anreichern: Wetter, Feiertage, Veranstaltungen, Wochentag

**TODO:**
1. Führen Sie `tage` mit `wetter` über `datum` zusammen (`how="left"`).
2. Legen Sie eine Spalte `feiertag` an: `1`, wenn das Datum in `feiertage["datum"]`
   vorkommt, sonst `0` (Tipp: `.isin(set(...))`, dann `.astype(int)`).
3. Legen Sie eine Spalte `event` an: `1`, wenn das Datum in **irgendeinem**
   Veranstaltungszeitraum liegt, sonst `0`. Anders als bei Feiertagen (einzelne Tage)
   müssen Sie hier für jede Zeile aus `veranstaltungen` prüfen, ob das Tagesdatum
   zwischen `von` und `bis` liegt (Tipp: eine Menge aller Veranstaltungstage bauen,
   indem Sie `pd.date_range(row["von"], row["bis"])` für jede Zeile in
   `veranstaltungen` erzeugen und die Ergebnisse zu einer einzigen Menge vereinigen).
4. Leiten Sie `wochentag` (`.dt.dayofweek`) und `monat` (`.dt.month`) ab.

In [ ]:
tage = ...  # TODO: merge von tage und wetter über 'datum', how="left"

tage["feiertag"] = ...  # TODO

event_tage = set()
for _, row in veranstaltungen.iterrows():
    event_tage |= set(pd.date_range(row["von"], row["bis"]))
tage["event"] = ...  # TODO: tage["datum"].isin(event_tage).astype(int)

tage["wochentag"] = ...  # TODO
tage["monat"] = ...  # TODO

tage[["datum", "fahrten", "temp_mittel_c", "niederschlag_mm", "feiertag", "event", "wochentag"]].head()

## Schritt 4 — Die Reihe anschauen, bevor wir prognostizieren

Ein Blick auf den Verlauf sagt oft mehr als jede Kennzahl: sehen wir die Saison, die wir
laut README erwarten?

**TODO:** Zeichnen Sie `tage["fahrten"]` über `tage["datum"]` als Linie.

In [ ]:
plt.figure(figsize=(12, 4))
...  # TODO: plt.plot(tage["datum"], tage["fahrten"])
plt.xlabel("Datum")
plt.ylabel("Fahrten pro Tag")
plt.title("VeloCity Tagesfahrten, September 2023 bis August 2026")
plt.show()

## Schritt 5 — Der Trainings-/Testsplit: zeitlich, nicht zufällig

**Das ist die wichtigste Regel dieses Notebooks, genauso wie im BINT-Notebook zur
Forecasting-Übung.** Wir legen einen Stichtag fest: alles davor ist Training, alles
danach ist Test. Wir nehmen die letzten knapp vier Monate (1. Mai bis 24. August 2026)
als Testzeitraum — genug Tage für ein verlässliches Fehlermaß, aber immer noch klar in
der Zukunft gegenüber dem Training.

**TODO:** Teilen Sie `tage` bei `datum < "2026-05-01"` (Training) und
`datum >= "2026-05-01"` (Test).

In [ ]:
split_datum = pd.Timestamp("2026-05-01")
train = ...  # TODO: tage, wo datum < split_datum
test = ...  # TODO: tage, wo datum >= split_datum

print("Training:", len(train), "Tage |", train["datum"].min(), "bis", train["datum"].max())
print("Test:", len(test), "Tage |", test["datum"].min(), "bis", test["datum"].max())
assert train["datum"].max() < test["datum"].min(), "Training und Test duerfen sich zeitlich nicht ueberschneiden!

## Schritt 6 — Klassisch: die saisonale Basislinie

Der einfachste sinnvolle Ansatz ganz ohne maschinelles Lernen: Für jeden Testtag nehmen
wir den **Mittelwert der letzten vier vergleichbaren Wochentage** aus den
Trainingsdaten. Ein Montag wird also anhand der letzten vier Trainingsmontage
geschätzt, ein Samstag anhand der letzten vier Trainingssamstage. Das fängt das
Wochenmuster ein (werktags anders als am Wochenende), ohne dass wir dafür irgendein
Modell trainieren müssen.

**TODO:**
1. Bilden Sie mit `train.groupby("wochentag")["fahrten"]` und
   `.apply(lambda s: s.tail(4).mean())` den Mittelwert der letzten vier Werte je
   Wochentag.
2. Übertragen Sie diesen Wert auf die Testtage, je nach ihrem Wochentag
   (`test["wochentag"].map(...)`).

In [ ]:
wochentag_mittel = ...  # TODO
vorhersage_klassisch = ...  # TODO: test["wochentag"].map(wochentag_mittel)

mape_klassisch = mape(test["fahrten"].values, vorhersage_klassisch.values)
print(f"MAPE klassische Basislinie: {mape_klassisch:.1f} %")

## Schritt 7 — Modern: Gradient Boosting

Gradient Boosting ist ein modernes Verfahren, das viele einfache Entscheidungsbäume
nacheinander baut — jeder neue Baum korrigiert gezielt die Fehler der bisherigen. Anders
als unsere klassische Basislinie kann es **mehrere Merkmale gleichzeitig** einbeziehen:
nicht nur den Wochentag, sondern auch Temperatur, Niederschlag, Monat, Feiertag und
Veranstaltungstag.

**TODO:**
1. Bilden Sie die Merkmalsliste `["wochentag", "monat", "temp_mittel_c",
   "niederschlag_mm", "feiertag", "event"]`.
2. Trainieren Sie `GradientBoostingRegressor(random_state=42)` auf den Trainingsdaten.
3. Sagen Sie die Testtage vorher und berechnen Sie den MAPE.

In [ ]:
merkmale = ["wochentag", "monat", "temp_mittel_c", "niederschlag_mm", "feiertag", "event"]

modell = ...  # TODO: GradientBoostingRegressor(random_state=42)
...  # TODO: modell.fit(train[merkmale], train["fahrten"])

vorhersage_modern = ...  # TODO: modell.predict(test[merkmale])
mape_modern = mape(test["fahrten"].values, vorhersage_modern)
print(f"MAPE Gradient Boosting: {mape_modern:.1f} %")

## Schritt 8 — Beide Prognosen im direkten Vergleich

**TODO:** Zeichnen Sie die tatsächlichen Testfahrten sowie beide Vorhersagen als Linien
übereinander.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test["datum"], test["fahrten"], label="Tatsächlich", linewidth=2, color="black")
plt.plot(test["datum"], vorhersage_klassisch, label=f"Klassisch (MAPE {mape_klassisch:.0f} %)", linestyle="--")
plt.plot(test["datum"], vorhersage_modern, label=f"Gradient Boosting (MAPE {mape_modern:.0f} %)", linestyle="--")
plt.xlabel("Datum")
plt.ylabel("Fahrten pro Tag")
plt.title("Prognosevergleich: klassisch vs. modern")
plt.legend()
plt.show()

print(f"Verbesserung durch Gradient Boosting: {mape_klassisch - mape_modern:.1f} Prozentpunkte "
      f"({(1 - mape_modern / mape_klassisch):.0%} relativer Fehlerabbau)")

## Schritt 9 — Interpretation

**Fragen zum Nachdenken:**
1. Welches der beiden eingesetzten Merkmale bei Gradient Boosting wirkt vermutlich am
   stärksten — Wetter oder Wochentag? Überlegen Sie anhand der README-Kennzahl (r ≈
   0,79 zur Temperatur), bevor Sie es nachprüfen (`modell.feature_importances_`).
2. Ein MAPE ist nie null, auch nicht beim besseren Modell. Was bedeutet ein MAPE von
   etwa 20 % konkret für die Disposition — für wie viele Fahrten sollte VeloCity an
   einem Tag mit prognostizierten 100 Fahrten realistisch vorsorgen?
3. **Rückbezug zu Notebook 3 (Clustering):** Die Merkmale hier gelten für ganz
   VeloCity. Wie könnte eine Prognose *je Stationscluster* (Pendler/Uni/Freizeit)
   noch genauer werden als eine Prognose für die Gesamtflotte?

In [ ]:
wichtigkeit = pd.Series(modell.feature_importances_, index=merkmale).sort_values(ascending=False)
wichtigkeit

*Ihre Antworten hier …*

## Zusammenfassung

In diesem Notebook haben Sie:
- Tagesdaten aus Einzelfahrten aggregiert und mit externen Quellen angereichert,
- einen zeitlich korrekten Trainings-/Testsplit angelegt — die wichtigste Regel bei
  jeder Zeitreihenprognose,
- eine klassische saisonale Basislinie ganz ohne maschinelles Lernen gebaut,
- ein modernes Verfahren (Gradient Boosting) mit mehreren Merkmalen trainiert,
- beide fair mit demselben Fehlermaß (MAPE) verglichen — und gesehen, dass "modern"
  hier tatsächlich einen spürbaren Unterschied macht, aber auch das bessere Modell
  nicht fehlerfrei ist.

**Damit sind alle vier Grundverfahren aus Block 05 durchgespielt.** Die nächste Station
im Kurs: Storytelling with Data (Block 06) — wie man genau diese Ergebnisse überzeugend
vor der Geschäftsführung präsentiert.